# tp-06 最終評価（test 2025、一度だけ）

**実行場所: ホスト PC（stg）。** tp-05e で決めたアンサンブル（`ensemble/ensemble_spec.json`）を、**2020〜2024 年で学習し、2025 年の全レースで評価する。test 2025 はここで初めて使う。**
モデル選択・パラメータ・合成の重みはすべて valid 2022〜2024 の OOF だけで決めたもの。結果を見てやり直さない（実行済みの印 `final_eval/TEST2025_EVALUATED.json` があると、再実行を拒否する）。

## 出す指標（勝率・連対率・複勝率）
- p_win（勝率）・p_top2（連対率）・p_top3（複勝率）それぞれの LogLoss・AUC・Brier・ECE
- 実用的な見え方: 予測 1 位の馬が勝つ割合、予測上位 2・3 頭に勝ち馬が入る割合、予測上位 3 頭のうち複勝圏に入った頭数の平均 ほか
- 較正（予測確率の十分位ごとの実測）、セグメント別（芝・ダート、距離帯、頭数、競馬場）
- 基準との比較: 一様予測（1/頭数）、LightGBM 既定、valid 2022〜2024 の OOF（安定性の確認）
- MLflow（`keiba-tp-final-eval`）に記録

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "6")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, gc, time
from datetime import datetime
from types import SimpleNamespace
import numpy as np
import pandas as pd
import mlflow
import psutil

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import ensemble as en, tp05_data, win_models as wm, zoo, zoo_runner
from src.pipeline.models.compute import detect_compute

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OOF_DIR = WORK / "oof/model_oof"
OUT = WORK / "final_eval"
OUT.mkdir(parents=True, exist_ok=True)
LOCK = OUT / "TEST2025_EVALUATED.json"
if LOCK.exists() and os.environ.get("TP06_FORCE") != "1":
    raise SystemExit(f"test 2025 は評価済みです（{LOCK}）。結果を見てやり直さないため、再実行しません。")
THREADS = detect_compute(reserve_cores=6).threads
SEEDS = [0, 1, 2, 3, 4]
mlflow.set_experiment("keiba-tp-final-eval")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
spec = json.loads((WORK / "ensemble/ensemble_spec.json").read_text(encoding="utf-8"))
best = json.loads((WORK / "hpo/best_params.json").read_text(encoding="utf-8"))["best"]
D = tp05_data.load_dataset(WORK)
df, meta = D["df"], D["meta"]
X = df[D["FEATS"]["base+S+C1"]]
dates = pd.to_datetime(meta["date"])
tr = np.flatnonzero(dates < pd.Timestamp("2025-01-01") - pd.Timedelta(days=10))
va = np.flatnonzero(meta["year"].to_numpy() == 2025)
test_fold = SimpleNamespace(fold_id="test_2025", train_index=tr, valid_index=va)
assert meta["year"].iloc[tr].max() <= 2024 and len(set(meta["race_id"].iloc[tr]) & set(meta["race_id"].iloc[va])) == 0
print(f"学習 {len(tr):,} 行（〜{dates.iloc[tr].max():%Y-%m-%d}） / test 2025 {len(va):,} 行・{meta['race_id'].iloc[va].nunique():,} レース / 構成: {spec['models']}")

In [ ]:
COMP = {
    "lgb_tuned": ("lgb_binary", zoo.tuned_params("lgb", best["lgb"]["params"])),
    "xgb_tuned": ("xgb_binary", zoo.tuned_params("xgb", best["xgb"]["params"])),
    "nn_tuned": ("nn", zoo.tuned_params("nn", best["nn"]["params"])),
    "lgb_softmax": ("lgb_softmax", {}),
    "linear": ("linear", {"epochs": 14}),
    "cat_binary": ("cat_binary", {"task_type": "GPU"}),
    "nn_s": ("nn", {"hidden": (128, 64), "epochs": 14}),
    "lgb_binary": ("lgb_binary", {}),
    "xgb_binary": ("xgb_binary", {"device": "cuda"}),
}
fin_all = meta["finish_position"].to_numpy()
race_all = meta["race_id"].to_numpy()
TEST = {}
for name in spec["models"]:
    recipe, params = COMP[name]
    gam = None
    if recipe in ("lgb_softmax",):          # p_top2・p_top3 を直接出さないモデルは、OOF で当てはめた γ を使う
        o = pd.read_parquet(OOF_DIR / f"{name}.parquet").merge(meta[["race_id", "horse_id", "finish_position"]], on=["race_id", "horse_id"])
        gam = (zoo.fit_gamma(o["race_id"].to_numpy(), o["p_win"].to_numpy(), o["finish_position"].to_numpy(), 2),
               zoo.fit_gamma(o["race_id"].to_numpy(), o["p_win"].to_numpy(), o["finish_position"].to_numpy(), 3))
    t0 = time.time()
    res, oof = zoo_runner.run_oof(name, recipe, X, meta, [test_fold], SEEDS, THREADS, params=dict(params), gammas=gam)
    TEST[name] = oof
    print(f"  {name:12s} test win {res['logloss_win'].iloc[0]:.5f} top2 {res['logloss_top2'].iloc[0]:.5f} top3 {res['logloss_top3'].iloc[0]:.5f} ({time.time() - t0:.0f}s) mem {psutil.Process().memory_info().rss / 1e9:.1f}GB")
    gc.collect()

In [ ]:
key = ["race_id", "horse_id"]
base = TEST[spec["models"][0]][key + ["year"]].copy()
for n, o in TEST.items():
    base = base.merge(o[key + ["p_win", "p_top2", "p_top3"]].rename(columns={f"p_{t}": f"{n}__{t}" for t in ("win", "top2", "top3")}), on=key)
base = base.merge(meta[key + ["finish_position"]], on=key)
race, fin = base["race_id"].to_numpy(), base["finish_position"].to_numpy()
TOT = {"win": 1.0, "top2": 2.0, "top3": 3.0}
final = {}
for t in TOT:
    Pm = np.stack([base[f"{n}__{t}"].to_numpy() for n in spec["models"]], axis=1)
    final[t] = wm.normalize_to_total(race, en.apply_spec(spec["targets"][t], Pm), TOT[t])
base["p_win"], base["p_top2"], base["p_top3"] = final["win"], final["top2"], final["top3"]
base.to_parquet(OUT / "test2025_predictions.parquet", index=False)
ens = zoo_runner.metrics_by_year(base[["year"]].reset_index(drop=True), final, fin, race, "ensemble")
display(ens.round(5))

In [ ]:
# 基準との比較
fields = base.groupby("race_id")["race_id"].transform("size").to_numpy()
uni = {"win": 1.0 / fields, "top2": np.minimum(2.0 / fields, 1.0), "top3": np.minimum(3.0 / fields, 1.0)}
rows = [{"model": "一様予測(1/頭数)", **{f"logloss_{t}": wm.binary_logloss(uni[t], (fin <= k).astype(float)) for t, k in (("win", 1), ("top2", 2), ("top3", 3))}}]
for n in spec["models"]:
    rows.append({"model": n, **{f"logloss_{t}": wm.binary_logloss(wm.normalize_to_total(race, base[f"{n}__{t}"].to_numpy(), TOT[t]), (fin <= k).astype(float)) for t, k in (("win", 1), ("top2", 2), ("top3", 3))}})
rows.append({"model": "アンサンブル(最終)", **{f"logloss_{t}": wm.binary_logloss(final[t], (fin <= k).astype(float)) for t, k in (("win", 1), ("top2", 2), ("top3", 3))}})
cmp_ = pd.DataFrame(rows).set_index("model")
oof_ens = pd.read_parquet(OOF_DIR / "ensemble.parquet").merge(meta[key + ["finish_position"]], on=key)
cmp_.loc["(参考) valid 2022〜2024 のアンサンブル(OOF)"] = [wm.binary_logloss(oof_ens[f"p_{t}"].to_numpy(), (oof_ens["finish_position"] <= k).astype(float).to_numpy()) for t, k in (("win", 1), ("top2", 2), ("top3", 3))]
display(cmp_.round(5)); cmp_.to_csv(OUT / "comparison.csv")

In [ ]:
# 実用的な指標
d = base.copy()
d["rank_win"] = d.groupby("race_id")["p_win"].rank(ascending=False, method="first")
d["rank_top3"] = d.groupby("race_id")["p_top3"].rank(ascending=False, method="first")
d["win"], d["top2"], d["top3"] = (d.finish_position == 1), (d.finish_position <= 2), (d.finish_position <= 3)
n_races = d.race_id.nunique()
top1 = d[d.rank_win == 1]
pick = {
    "予測1位の馬が勝つ割合（勝率）": top1["win"].mean(),
    "予測1位の馬が連対(2着以内)する割合": top1["top2"].mean(),
    "予測1位の馬が複勝圏(3着以内)に入る割合": top1["top3"].mean(),
    "勝ち馬が予測上位2頭に入る割合": d[(d.rank_win <= 2) & d.win].race_id.nunique() / n_races,
    "勝ち馬が予測上位3頭に入る割合": d[(d.rank_win <= 3) & d.win].race_id.nunique() / n_races,
    "予測上位3頭(複勝確率順)のうち複勝圏に入った頭数の平均（最大3）": d[d.rank_top3 <= 3].groupby("race_id")["top3"].sum().mean(),
    "予測上位3頭(複勝確率順)が全員複勝圏のレース割合": (d[d.rank_top3 <= 3].groupby("race_id")["top3"].sum() == 3).mean(),
    "一様予測の1位的中率(参考)": float((1 / fields).mean()),
}
P = pd.Series(pick, name="test 2025"); display(P.round(4)); P.to_csv(OUT / "practical_metrics.csv")
# 予測1位の確率帯別（予測勝率が高いほど実際に勝つか）
d1 = top1.copy(); d1["bin"] = pd.qcut(d1["p_win"], 5, duplicates="drop")
tab = d1.groupby("bin", observed=True).agg(races=("win", "size"), 予測勝率=("p_win", "mean"), 実際の勝率=("win", "mean"), 実際の連対率=("top2", "mean"), 実際の複勝率=("top3", "mean"))
display(tab.round(3)); tab.to_csv(OUT / "top1_by_confidence.csv")

In [ ]:
# 較正: 予測確率の十分位ごとの平均予測と実測
def reliability(p, y, q=10):
    b = pd.qcut(p, q, duplicates="drop")
    return pd.DataFrame({"p": p, "y": y, "b": b}).groupby("b", observed=True).agg(n=("y", "size"), 予測=("p", "mean"), 実測=("y", "mean"))


REL = {t: reliability(final[t], (fin <= k).astype(float)) for t, k in (("win", 1), ("top2", 2), ("top3", 3))}
for t, r in REL.items():
    print(f"{t}: ECE {np.average((r['予測'] - r['実測']).abs(), weights=r['n']):.5f}"); display(r.round(4))
    r.to_csv(OUT / f"reliability_{t}.csv")

In [ ]:
# セグメント別の LogLoss（悪化している区分がないか）
seg = base.merge(df[["race_id", "horse_id", "surface", "distance", "field_size", "venue_code"]].astype({"race_id": str, "horse_id": str}), on=key)
seg["距離帯"] = pd.cut(seg["distance"], [0, 1300, 1700, 2100, 2600, 9999], labels=["〜1300", "1400〜1700", "1800〜2100", "2200〜2600", "2700〜"])
seg["頭数"] = pd.cut(seg["field_size"], [0, 11, 14, 99], labels=["〜11頭", "12〜14頭", "15頭〜"])
rows = []
for col in ("surface", "距離帯", "頭数", "venue_code"):
    for v, g in seg.groupby(col, observed=True):
        f = g["finish_position"].to_numpy()
        rows.append({"区分": col, "値": str(v), "行数": len(g), **{f"logloss_{t}": wm.binary_logloss(g[f"p_{t}"].to_numpy(), (f <= k).astype(float)) for t, k in (("win", 1), ("top2", 2), ("top3", 3))},
                     "1位的中": float((g.assign(r=g.groupby("race_id")["p_win"].rank(ascending=False, method="first")).query("r==1")["finish_position"] == 1).mean())})
S = pd.DataFrame(rows)
display(S.round(4)); S.to_csv(OUT / "segments.csv", index=False)

In [ ]:
res = ens.iloc[0].to_dict()
report = {"git_commit": GIT_COMMIT, "evaluated_at": datetime.now().isoformat(timespec="seconds"), "train_rows": int(len(tr)), "test_rows": int(len(va)), "test_races": int(n_races),
          "models": spec["models"], "methods": {t: spec["targets"][t]["method"] for t in TOT}, "metrics": res, "practical": {k: float(v) for k, v in pick.items()},
          "comparison": cmp_.round(6).reset_index().to_dict("records"), "seeds": SEEDS}
(OUT / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
with mlflow.start_run(run_name="tp-06-final-test2025", tags={"notebook": "tp-06", "git_commit": GIT_COMMIT, "test_year": "2025", "train": "2020-2024"}):
    mlflow.log_params({"models": ",".join(spec["models"]), **{f"method_{t}": spec["targets"][t]["method"] for t in TOT}, "seeds": str(SEEDS)})
    mlflow.log_metrics({k: float(v) for k, v in res.items() if k not in ("model", "fold")})
    mlflow.log_metrics({f"practical_{i}": float(v) for i, v in enumerate(pick.values())})
    for f_ in ("report.json", "comparison.csv", "practical_metrics.csv", "segments.csv"):
        mlflow.log_artifact(str(OUT / f_), "final_eval")
LOCK.write_text(json.dumps({"evaluated_at": report["evaluated_at"], "git_commit": GIT_COMMIT, "note": "test 2025 を評価済み。結果を見てやり直さない。"}, ensure_ascii=False), encoding="utf-8")
print("test 2025 の評価を完了しました:", {k: round(float(v), 5) for k, v in res.items() if k.startswith("logloss") or k.startswith("auc")})